In [ ]:
import cv2
import numpy as np
import pytesseract
from passporteye import read_mrz
from langdetect import detect
from deep_translator import GoogleTranslator
import re

def display_image(title, img):
    cv2.imshow(title, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def deskew_image(image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        angles = [np.arctan2(y2 - y1, x2 - x1) for [[x1, y1, x2, y2]] in lines]
        median_angle = np.median(angles)
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, np.degrees(median_angle), 1.0)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return image

def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 9, 75, 75) 
    adaptive_thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 2
    )

    kernel = np.ones((1, 1), np.uint8)
    processed = cv2.morphologyEx(adaptive_thresh, cv2.MORPH_CLOSE, kernel)

    return processed

def extract_mrz(image_path):

    mrz = read_mrz(image_path)
    if mrz:
        return mrz.to_dict()
    return None

def extract_text(image):

    config = "--oem 3 --psm 6"
    text = pytesseract.image_to_string(image, config=config)
    return text

def correct_text(text):
    
    detected_lang = detect(text)
    corrected_text = GoogleTranslator(source=detected_lang, target="en").translate(text)

    
    corrected_text = re.sub(r'\n+', '\n', corrected_text).strip()
    return corrected_text

def format_mrz_data(mrz_data):
    formatted_data = {
        "Passport Type": mrz_data.get("type", "").replace("<", ""),
        "Country": mrz_data.get("country", "").replace("<", ""),
        "Passport Number": mrz_data.get("number", "").replace("<", ""),
        "Surname": mrz_data.get("surname", "").replace("<", ""),
        "Given Names": mrz_data.get("names", "").replace("<", ""),
        "Date of Birth": mrz_data.get("date_of_birth", ""),
        "Sex": mrz_data.get("sex", ""),
        "Nationality": mrz_data.get("nationality", ""),
        "Expiration Date": mrz_data.get("expiration_date", ""),
        "Personal Number": mrz_data.get("personal_number", ""),
        "MRZ Raw Text": mrz_data.get("raw_text", ""),
    }
    return formatted_data

def fill_missing_data(mrz_data, extracted_text):
    
    filled_data = {}

    filled_data["Passport Number"] = mrz_data["Passport Number"] if mrz_data["Passport Number"] else re.search(r"\b[A-Z0-9]{7,9}\b", extracted_text)
    filled_data["Date of Birth"] = mrz_data["Date of Birth"] if mrz_data["Date of Birth"] else re.search(r"\b\d{2}/\d{2}/\d{4}\b", extracted_text)
    filled_data["Expiration Date"] = mrz_data["Expiration Date"] if mrz_data["Expiration Date"] else re.search(r"\b\d{2}/\d{2}/\d{4}\b", extracted_text)
    filled_data["Surname"] = mrz_data["Surname"] if mrz_data["Surname"] else re.search(r"([A-Z]+)", extracted_text)
    filled_data["Given Names"] = mrz_data["Given Names"] if mrz_data["Given Names"] else re.search(r"([A-Z]+\s[A-Z]+)", extracted_text)

    return filled_data

def process_passport(image_path):

    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Unable to load image. Check file path: {image_path}")
        return

    image = deskew_image(image)
    display_image("Deskewed Image", image)

    processed = preprocess_image(image)
    display_image("Preprocessed Image", processed)

    mrz_data = extract_mrz(image_path)  
    extracted_text = extract_text(processed)
    corrected_text = correct_text(extracted_text)

    if mrz_data:
        formatted_mrz = format_mrz_data(mrz_data)
        filled_data = fill_missing_data(formatted_mrz, extracted_text)

        print("\n--- Extracted Passport Details ---")
        for key, value in filled_data.items():
            print(f"{key}: {value}")

        print("\n--- MRZ Raw Text ---")
        print(mrz_data["raw_text"])
    else:
        print("\nNo MRZ Data Found.")

    print("\n--- Extracted OCR Text ---")
    print(extracted_text)

    print("\n--- Corrected Text ---")
    print(corrected_text)

if __name__ == "__main__":
    image_path = "passport/1.jpg" 
    process_passport(image_path)


c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:37: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage_io.imread(file, as_gray=self.as_gray, plugin='imageio')
c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:89: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  m = morphology.square(self.square_size)



--- Extracted Passport Details ---
Passport Number: W8129257
Date of Birth: 980111
Expiration Date: 330117
Surname: RAVI
Given Names: GUNASEKARK K KKKKKKKKKKKKKKKKKKKK

--- MRZ Raw Text ---
P<INDRAVI<<GUNASEKARK<K<KKKKKKKKKKKKKKKKKKKKKK
W8129257<81ND9801118M330117721C5002044323<28

--- Extracted OCR Text ---
eB lire Hr te re ee wie Ahad,
an - Ease mene nme” ee eee” Me a
melt tr t-—* “<-> waineser a
-_ a ‘ i amewene oe _ a
ak 4 4 penctoaae . ~
. t { ——_ Oe . .
ae yee) o% :
en h(—— * :
Pal On searwn se, tenis Bane “ .
eet - Ob peceneneee . Ye
144. soimearone - we .
: Re py- ened owen ee
“eth c sarevsreas , tremsness we
fom . . .
.
PC IMD RAVE CCE UNAS ER AREK KEEL LEER EEE EEEL EERE
WEIZIZSPCEIMOOMOVITSMSSOVIPTZICSOOZOAASZS<728
foo


--- Corrected Text ---
eB lire Hr te re ee wie Ahad,
an - Ease mene nme” ee eee” Me a
melt tr t-—* “<-> waineser a
-_ a ‘ i amewene oe _ a
ak 4 4 penctoaae . ~
. t { ——_ Oe . .
ae yee) o% :
en h(—— * :
Pal On searwn se, tenis Bane “ .
eet - Ob peceneneee . 